In [ ]:
pip install langchain langgraph langchain_community langchain_openai

In [2]:
import langchain
import langgraph
import langchain_community
import langchain_openai

In [3]:
import langchain_community

help(langchain_community)

Help on package langchain_community:

NAME
    langchain_community - Main entrypoint into package.

PACKAGE CONTENTS
    adapters (package)
    agent_toolkits (package)
    agents (package)
    cache
    callbacks (package)
    chains (package)
    chat_loaders (package)
    chat_message_histories (package)
    chat_models (package)
    cross_encoders (package)
    docstore (package)
    document_compressors (package)
    document_loaders (package)
    document_transformers (package)
    embeddings (package)
    example_selectors (package)
    graph_vectorstores (package)
    graphs (package)
    indexes (package)
    llms (package)
    memory (package)
    output_parsers (package)
    query_constructors (package)
    retrievers (package)
    storage (package)
    tools (package)
    utilities (package)
    utils (package)
    vectorstores (package)

VERSION
    0.3.31

FILE
    /opt/anaconda3/lib/python3.11/site-packages/langchain_community/__init__.py




In [ ]:
pip install langchain_community

In [4]:
import langchain

#dir(langchain)
print(langchain.__version__)

0.3.27


In [6]:
import langchain.tools

help(langchain.tools)

Help on package langchain.tools in langchain:

NAME
    langchain.tools - **Tools** are classes that an Agent uses to interact with the world.

DESCRIPTION
    Each tool has a **description**. Agent uses the description to choose the right
    tool for the job.
    
    **Class hierarchy:**
    
    .. code-block::
    
        ToolMetaclass --> BaseTool --> <name>Tool  # Examples: AIPluginTool, BaseGraphQLTool
                                       <name>      # Examples: BraveSearch, HumanInputRun
    
    **Main helpers:**
    
    .. code-block::
    
        CallbackManagerForToolRun, AsyncCallbackManagerForToolRun

PACKAGE CONTENTS
    ainetwork (package)
    amadeus (package)
    arxiv (package)
    azure_cognitive_services (package)
    base
    bearly (package)
    bing_search (package)
    brave_search (package)
    clickup (package)
    convert_to_openai
    dataforseo_api_search (package)
    ddg_search (package)
    e2b_data_analysis (package)
    edenai (package)
    elev

In [7]:
#Create a Tool (Example: get_weather)

from langchain.tools import tool

@tool   # decorator
def get_weather(city: str) -> str:
    """Returns a mock weather report for the given city."""
    return f"It's always sunny in {city} with 25°C!"

In [9]:
#help(tool)

In [8]:
#Initialize GPT Model (via LangChain)

from langchain.chat_models import init_chat_model

model4 = init_chat_model("gpt-4.1", model_provider="openai")

In [9]:
#Initialize GPT Model (via LangChain)

from langchain.chat_models import init_chat_model

model5 = init_chat_model("gpt-5", model_provider="openai")

In [ ]:
help(model5)

In [11]:
#Create the ReAct Agent (with Tool Binding)
from langgraph.prebuilt import create_react_agent

tools = [get_weather]
agent_executor = create_react_agent(model=model5, tools=tools)

In [12]:
# Run the Agent Without Memory (Single Turn)

input_message = {"role": "user", "content": "What is the weather in Pune?"}
response = agent_executor.invoke({"messages": [input_message]})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the weather in Pune?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_F9BG5rkoz2yG3cGqd52sNBvV)
 Call ID: call_F9BG5rkoz2yG3cGqd52sNBvV
  Args:
    city: Pune
================================= Tool Message =================================
Name: get_weather

It's always sunny in Pune with 25°C!
================================== Ai Message ==================================

It's always sunny in Pune with 25°C!


In [13]:
#Add Memory for Multi-Turn Agent

from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
agent_executor = create_react_agent(model=model5, tools=tools, checkpointer=memory)

config = {"configurable": {"thread_id": "weather-001"}}

# Turn 1
response1=agent_executor.invoke({"messages": [{"role": "user", "content": "My city is Nagpur"}]}, config)

for msg in response1["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My city is Nagpur
================================== Ai Message ==================================

Got it—Nagpur. What would you like help with?
- Current (mock) weather
- Places to visit or eat
- Local transport/commute tips
- Day trip ideas
- Anything else

Want me to fetch the (mock) current weather for Nagpur?


In [14]:
# Turn 2
response2 = agent_executor.invoke({"messages": [{"role": "user", "content": "What's the weather where I live?"}]}, config)

for msg in response2["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My city is Nagpur
================================== Ai Message ==================================

Got it—Nagpur. What would you like help with?
- Current (mock) weather
- Places to visit or eat
- Local transport/commute tips
- Day trip ideas
- Anything else

Want me to fetch the (mock) current weather for Nagpur?
================================ Human Message =================================

What's the weather where I live?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_yj0CWAy3ErpgDotoRBAvsLSd)
 Call ID: call_yj0CWAy3ErpgDotoRBAvsLSd
  Args:
    city: Nagpur
================================= Tool Message =================================
Name: get_weather

It's always sunny in Nagpur with 25°C!
================================== Ai Message ==================================

Here’s your (mock) local weather for Nagpur: Sunny, around 25

In [25]:
import json
from typing import Dict
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
import pprint

# -----------------------------
# Mock Telecom Tool Functions
# -----------------------------

@tool
def get_customer_profile(customer_id: str) -> str:
    """
    Fetch customer profile details from CRM/billing system.
    """
    data = {
        "customer_id": customer_id,
        "name": "Ravi Patil",
        "location": "Pune",
        "plan": "5G Unlimited Plus",
        "device_type": "Android 5G",
        "recent_complaints": 2
    }
    return json.dumps(data, indent=2)

@tool
def get_network_metrics(location: str) -> str:
    """
    Fetch network KPIs for the location.
    """
    data = {
        "location": location,
        "latency_ms": 210,
        "packet_loss_percent": 8,
        "signal_strength_dbm": -89,
        "tower_utilization_percent": 93,
        "network_status": "Congested"
    }
    return json.dumps(data, indent=2)

@tool
def get_device_health(device_type: str) -> str:
    """
    Simulate device-side checks.
    """
    data = {
        "device_type": device_type,
        "device_issue_probability": "Low",
        "sim_status": "Healthy",
        "battery_mode": "Normal"
    }
    return json.dumps(data, indent=2)

# -----------------------------
# LLM
# -----------------------------

# llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -----------------------------
# 1) LLM-only Diagnosis
# -----------------------------

def llm_only_demo():
    prompt = """
    You are a telecom support assistant.
    A customer says: "My mobile internet is very slow in Pune since morning."
    Give diagnosis and recommendation.
    """
    response = model5.invoke(prompt)
    print("\n===== LLM ONLY OUTPUT =====\n")
    print(response.pretty_print())

# -----------------------------
# 2) Agent-based Diagnosis
# -----------------------------

tools = [get_customer_profile, get_network_metrics, get_device_health]

agent = initialize_agent(
    tools=tools,
    llm=model4,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)

def agent_demo():
    query = """
    Diagnose why customer CUST_1001 is facing slow mobile internet in Pune since morning.
    Use available tools.
    Then provide:
    1. Root cause
    2. Confidence
    3. Recommended action
    4. Customer-facing response
    """
    print("\n===== AGENT OUTPUT =====\n")
    result = agent.run(query)
    pprint.pprint(result)

# -----------------------------
# Run Both
# -----------------------------

if __name__ == "__main__":
    llm_only_demo()
    #agent_demo()


===== LLM ONLY OUTPUT =====

================================== Ai Message ==================================

Thanks for reporting this. Based on “since morning” and location-specific impact (Pune), the most likely causes are:
- Temporary network degradation: planned maintenance, site/backhaul issue, or unusual cell congestion in your area.
- Coverage/technology mismatch: weak indoor signal or 5G band with poor indoor penetration causing slow/unstable data; device falling back between 5G/4G.
- Account/data cap: Fair-usage limit (FUP) reached leading to throttling.
- Device/SIM/APN glitch.

What to try now (in order):
1) Refresh your radio connection
- Toggle Airplane Mode on for 30 seconds, then off.
- Restart the phone.

2) Lock to 4G temporarily
- Settings > Mobile/Cellular > Preferred network type: select 4G/LTE only (turn off 5G for now). Then test.

3) Check data/FUP
- In your operator’s app or via USSD/portal, confirm you haven’t hit a speed cap. If capped, add a data booster/t

In [26]:
import json
from typing import TypedDict, Optional, Dict, Any

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END

# =========================================================
# 1. Mock Telecom Tools
# =========================================================

def get_customer_profile(customer_id: str) -> Dict[str, Any]:
    """
    Simulates fetching customer profile from CRM/BSS.
    """
    return {
        "customer_id": customer_id,
        "name": "Ravi Patil",
        "location": "Pune",
        "plan": "5G Unlimited Plus",
        "device_type": "Android 5G",
        "recent_complaints": 2,
        "customer_segment": "Premium"
    }


def get_network_metrics(location: str) -> Dict[str, Any]:
    """
    Simulates fetching network KPIs from OSS/NMS.
    """
    return {
        "location": location,
        "latency_ms": 210,
        "packet_loss_percent": 8,
        "signal_strength_dbm": -89,
        "tower_utilization_percent": 93,
        "network_status": "Congested",
        "outage_detected": False
    }


def get_device_health(device_type: str) -> Dict[str, Any]:
    """
    Simulates device-side health checks.
    """
    return {
        "device_type": device_type,
        "device_issue_probability": "Low",
        "sim_status": "Healthy",
        "battery_mode": "Normal",
        "os_network_issue": False
    }



In [27]:
# =========================================================
# 2. LLM Setup
# =========================================================

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

# =========================================================
# 3. LLM-Only Demo
# =========================================================

def llm_only_demo():
    prompt = """
You are a telecom support assistant.

Customer complaint:
"My mobile internet is very slow in Pune since morning."

Provide:
1. Likely issue
2. Recommendation
3. Customer response

Do not assume access to internal telecom tools.
"""
    response = llm.invoke([HumanMessage(content=prompt)])
    print("\n" + "=" * 70)
    print("LLM-ONLY DIAGNOSIS")
    print("=" * 70)
    print(response.content)

In [28]:
if __name__ == "__main__":
    llm_only_demo()


LLM-ONLY DIAGNOSIS
1. **Likely Issue**: The slow mobile internet in Pune could be due to several factors, such as network congestion, maintenance work in the area, or a temporary outage affecting service quality.

2. **Recommendation**: I recommend trying the following steps to improve your mobile internet speed:
   - Restart your mobile device to refresh the network connection.
   - Check if you have a strong signal; if not, try moving to a different location.
   - Disable any background apps that may be using data.
   - Switch between 4G and 3G settings in your mobile network settings to see if one works better than the other.
   - If the issue persists, consider turning on Airplane mode for a few seconds and then turning it off to reconnect to the network.

3. **Customer Response**: "Thank you for the suggestions! I will try restarting my phone and checking my signal strength. If the problem continues, I’ll let you know."


In [30]:
# =========================================================
# 4. LangGraph State Definition
# =========================================================

class TelecomAgentState(TypedDict, total=False):
    complaint: str
    customer_id: str

    parsed_issue: str
    location: str

    customer_profile: Dict[str, Any]
    network_metrics: Dict[str, Any]
    device_health: Dict[str, Any]

    diagnosis: str
    confidence: str
    recommended_action: str
    customer_response: str


In [31]:

# =========================================================
# 5. LangGraph Nodes
# =========================================================

def parse_issue_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Extract complaint details using the LLM.
    """
    complaint = state["complaint"]

    prompt = f"""
You are a telecom AI assistant.

Extract the following from this complaint:
- issue_type
- location
- urgency

Complaint:
"{complaint}"

Return ONLY JSON in this format:
{{
  "issue_type": "...",
  "location": "...",
  "urgency": "..."
}}
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    try:
        parsed = json.loads(response.content)
    except Exception:
        parsed = {
            "issue_type": "slow internet",
            "location": "Pune",
            "urgency": "medium"
        }

    return {
        "parsed_issue": parsed.get("issue_type", "slow internet"),
        "location": parsed.get("location", "Pune")
    }


def fetch_customer_profile_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Fetch customer profile.
    """
    customer_id = state["customer_id"]
    profile = get_customer_profile(customer_id)
    return {"customer_profile": profile}


def fetch_network_metrics_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Fetch network KPIs for location.
    """
    location = state["location"]
    metrics = get_network_metrics(location)
    return {"network_metrics": metrics}


def fetch_device_health_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Fetch device diagnostic info.
    """
    device_type = state["customer_profile"]["device_type"]
    health = get_device_health(device_type)
    return {"device_health": health}


def diagnose_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Use LLM + fetched telecom signals to create grounded diagnosis.
    """
    complaint = state["complaint"]
    customer_profile = state["customer_profile"]
    network_metrics = state["network_metrics"]
    device_health = state["device_health"]

    prompt = f"""
You are an expert telecom network diagnosis agent.

Customer complaint:
{complaint}

Customer profile:
{json.dumps(customer_profile, indent=2)}

Network metrics:
{json.dumps(network_metrics, indent=2)}

Device health:
{json.dumps(device_health, indent=2)}

Based on the above data, determine:
1. Most likely root cause
2. Confidence level (High / Medium / Low)
3. Best next action for telecom operations team

Return ONLY JSON in this format:
{{
  "diagnosis": "...",
  "confidence": "...",
  "recommended_action": "..."
}}
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    try:
        result = json.loads(response.content)
    except Exception:
        result = {
            "diagnosis": "High probability of local tower congestion causing degraded mobile internet performance.",
            "confidence": "High",
            "recommended_action": "Raise congestion alert to network operations team and inform customer of temporary service degradation."
        }

    return {
        "diagnosis": result.get("diagnosis", ""),
        "confidence": result.get("confidence", ""),
        "recommended_action": result.get("recommended_action", "")
    }


def generate_customer_response_node(state: TelecomAgentState) -> TelecomAgentState:
    """
    Create customer-facing message from diagnosis.
    """
    diagnosis = state["diagnosis"]
    confidence = state["confidence"]
    action = state["recommended_action"]
    customer_profile = state["customer_profile"]

    prompt = f"""
You are a telecom support assistant.

Create a professional, short customer-facing response.

Context:
- Customer Name: {customer_profile.get("name")}
- Diagnosis: {diagnosis}
- Confidence: {confidence}
- Action: {action}

Requirements:
- Keep response polite
- Do not expose internal technical jargon too much
- Mention that team is checking if appropriate
- Maximum 80 words
"""
    response = llm.invoke([HumanMessage(content=prompt)])

    return {
        "customer_response": response.content
    }



In [32]:
# =========================================================
# 6. Build LangGraph Workflow
# =========================================================

def build_graph():
    workflow = StateGraph(TelecomAgentState)

    # Add nodes
    workflow.add_node("parse_issue", parse_issue_node)
    workflow.add_node("fetch_customer_profile", fetch_customer_profile_node)
    workflow.add_node("fetch_network_metrics", fetch_network_metrics_node)
    workflow.add_node("fetch_device_health", fetch_device_health_node)
    workflow.add_node("diagnose", diagnose_node)
    workflow.add_node("generate_customer_response", generate_customer_response_node)

    # Set entry point
    workflow.set_entry_point("parse_issue")

    # Add edges
    workflow.add_edge("parse_issue", "fetch_customer_profile")
    workflow.add_edge("fetch_customer_profile", "fetch_network_metrics")
    workflow.add_edge("fetch_network_metrics", "fetch_device_health")
    workflow.add_edge("fetch_device_health", "diagnose")
    workflow.add_edge("diagnose", "generate_customer_response")
    workflow.add_edge("generate_customer_response", END)

    return workflow.compile()


In [33]:
# =========================================================
# 7. Run Agent Demo
# =========================================================

def agentic_langgraph_demo():
    graph = build_graph()

    initial_state = {
        "complaint": "My mobile internet is very slow in Pune since morning.",
        "customer_id": "CUST_1001"
    }

    result = graph.invoke(initial_state)

    print("\n" + "=" * 70)
    print("LANGGRAPH AGENT DIAGNOSIS")
    print("=" * 70)

    print("\nParsed Issue:")
    print(result.get("parsed_issue"))

    print("\nLocation:")
    print(result.get("location"))

    print("\nCustomer Profile:")
    print(json.dumps(result.get("customer_profile", {}), indent=2))

    print("\nNetwork Metrics:")
    print(json.dumps(result.get("network_metrics", {}), indent=2))

    print("\nDevice Health:")
    print(json.dumps(result.get("device_health", {}), indent=2))

    print("\nDiagnosis:")
    print(result.get("diagnosis"))

    print("\nConfidence:")
    print(result.get("confidence"))

    print("\nRecommended Action:")
    print(result.get("recommended_action"))

    print("\nCustomer Response:")
    print(result.get("customer_response"))

In [34]:
# =========================================================
# 8. Main
# =========================================================

if __name__ == "__main__":
    #llm_only_demo()
    agentic_langgraph_demo()


LANGGRAPH AGENT DIAGNOSIS

Parsed Issue:
slow mobile internet

Location:
Pune

Customer Profile:
{
  "customer_id": "CUST_1001",
  "name": "Ravi Patil",
  "location": "Pune",
  "plan": "5G Unlimited Plus",
  "device_type": "Android 5G",
  "recent_complaints": 2,
  "customer_segment": "Premium"
}

Network Metrics:
{
  "location": "Pune",
  "latency_ms": 210,
  "packet_loss_percent": 8,
  "signal_strength_dbm": -89,
  "tower_utilization_percent": 93,
  "network_status": "Congested",
  "outage_detected": false
}

Device Health:
{
  "device_type": "Android 5G",
  "device_issue_probability": "Low",
  "sim_status": "Healthy",
  "battery_mode": "Normal",
  "os_network_issue": false
}

Diagnosis:
The slow mobile internet in Pune is likely due to network congestion, as indicated by high tower utilization (93%) and significant packet loss (8%).

Confidence:
High

Recommended Action:
Increase capacity on the affected tower or redistribute load to nearby towers to alleviate congestion.

Customer 

A. LLM-only flow
The LLM gets only this complaint:
“My mobile internet is very slow in Pune since morning.”
So it gives a generic answer such as:
•	restart device
•	toggle airplane mode
•	move to another area
•	check plan validity
This is useful, but not grounded in telecom telemetry.


B. LangGraph agent flow
The LangGraph workflow does this:
1.	Parse complaint
2.	Fetch customer profile
3.	Fetch network metrics
4.	Fetch device health
5.	Diagnose using real context
6.	Generate customer response
So the diagnosis becomes evidence-based.

LangGraph Flow
Start
  ↓
parse_issue
  ↓
fetch_customer_profile
  ↓
fetch_network_metrics
  ↓
fetch_device_health
  ↓
diagnose
  ↓
generate_customer_response
  ↓
End

Why LangGraph is Better Here

For telecom diagnosis, LangGraph is useful because it gives:
•	structured workflow
•	clear node-by-node orchestration
•	easy extensibility
•	better observability
•	controlled enterprise flow
This fits better than a plain chatbot when the team wants:
•	repeatable diagnosis flow
•	tool orchestration
•	auditability
•	future escalation logic

LLM-only
•	understands language
•	gives general suggestions
•	no access to network telemetry
•	no operational workflow


LangGraph Agent
•	reads complaint
•	fetches telecom signals
•	reasons on actual metrics
•	suggests operational next step

A plain LLM can discuss a slow-network complaint, but it cannot reliably diagnose telecom issues without structured access to customer, network, and device data. LangGraph helps convert the interaction into an Agentic AI workflow by orchestrating diagnostic steps, tool usage, and grounded response generation.

Lab: Build a Basic Chatbot → 
Convert it into a Telecom Agent
Lab Objective
Participants will:
•	build a basic LLM chatbot
•	understand its limitation for telecom diagnosis
•	convert it into a tool-using telecom agent
•	compare chatbot vs agent

Build a Basic Chatbot 

Goal
Create a simple chatbot that responds to telecom customer complaints.

In [35]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def basic_chatbot(user_query: str):
    messages = [
        SystemMessage(content="You are a helpful telecom customer support chatbot."),
        HumanMessage(content=user_query)
    ]
    response = llm.invoke(messages)
    return response.content

if __name__ == "__main__":
    query = "My mobile internet is very slow in Pune since morning."
    result = basic_chatbot(query)
    print("\n=== BASIC CHATBOT RESPONSE ===\n")
    print(result)


=== BASIC CHATBOT RESPONSE ===

I'm sorry to hear that you're experiencing slow mobile internet in Pune. Here are a few steps you can try to troubleshoot the issue:

1. **Check Network Coverage**: Ensure that you are in an area with good network coverage. You can check your signal strength on your device.

2. **Restart Your Device**: Sometimes, simply restarting your phone can resolve connectivity issues.

3. **Toggle Airplane Mode**: Turn on Airplane Mode for a few seconds and then turn it off. This can help reset your connection to the network.

4. **Check for Outages**: There may be a temporary network outage in your area. You can check your service provider's website or social media pages for any announcements.

5. **Clear Cache**: If you're using a specific app that is slow, try clearing its cache or data.

6. **Limit Background Data**: Ensure that no apps are using excessive data in the background.

7. **Update Your Device**: Make sure your device's software is up to date, as up

Expected Behavior
The chatbot will usually respond with generic guidance such as:
•	restart device
•	toggle airplane mode
•	check signal strength
•	move to another location
•	contact support

This chatbot is useful for conversation, but:
•	it does not know customer profile
•	it does not know network metrics
•	it does not know tower congestion
•	it cannot take diagnostic action


Convert to Telecom Agent 

Goal
Enhance the chatbot by adding telecom tools.
The agent will:
•	fetch customer profile
•	fetch network metrics
•	fetch device health
•	diagnose likely root cause


In [36]:
import json
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import initialize_agent, AgentType

# -----------------------------
# LLM
# -----------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# -----------------------------
# Mock Telecom Tools
# -----------------------------
@tool
def get_customer_profile(customer_id: str) -> str:
    """Fetch telecom customer profile."""
    data = {
        "customer_id": customer_id,
        "name": "Ravi Patil",
        "location": "Pune",
        "plan": "5G Unlimited Plus",
        "device_type": "Android 5G",
        "recent_complaints": 2
    }
    return json.dumps(data, indent=2)

@tool
def get_network_metrics(location: str) -> str:
    """Fetch telecom network metrics for a location."""
    data = {
        "location": location,
        "latency_ms": 210,
        "packet_loss_percent": 8,
        "signal_strength_dbm": -89,
        "tower_utilization_percent": 93,
        "network_status": "Congested"
    }
    return json.dumps(data, indent=2)

@tool
def get_device_health(device_type: str) -> str:
    """Fetch device-side health information."""
    data = {
        "device_type": device_type,
        "device_issue_probability": "Low",
        "sim_status": "Healthy",
        "battery_mode": "Normal"
    }
    return json.dumps(data, indent=2)

# -----------------------------
# Agent
# -----------------------------
tools = [get_customer_profile, get_network_metrics, get_device_health]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,
    verbose=True
)

def telecom_agent_demo():
    query = """
    Diagnose why customer CUST_1001 is facing slow mobile internet in Pune since morning.

    Use the available tools.
    Provide:
    1. Root cause
    2. Confidence
    3. Recommended action
    4. Customer-facing response
    """
    result = agent.run(query)
    return result

if __name__ == "__main__":
    output = telecom_agent_demo()
    print("\n=== TELECOM AGENT RESPONSE ===\n")
    print(output)



> Entering new AgentExecutor chain...

Invoking: `get_customer_profile` with `{'customer_id': 'CUST_1001'}`


{
  "customer_id": "CUST_1001",
  "name": "Ravi Patil",
  "location": "Pune",
  "plan": "5G Unlimited Plus",
  "device_type": "Android 5G",
  "recent_complaints": 2
}
Invoking: `get_network_metrics` with `{'location': 'Pune'}`


{
  "location": "Pune",
  "latency_ms": 210,
  "packet_loss_percent": 8,
  "signal_strength_dbm": -89,
  "tower_utilization_percent": 93,
  "network_status": "Congested"
}
Invoking: `get_device_health` with `{'device_type': 'Android 5G'}`


{
  "device_type": "Android 5G",
  "device_issue_probability": "Low",
  "sim_status": "Healthy",
  "battery_mode": "Normal"
}### Diagnosis of Slow Mobile Internet for Customer CUST_1001

1. **Root Cause**: 
   - The network metrics for Pune indicate that the network is congested, with a tower utilization of 93%. Additionally, there is a high latency of 210 ms and a packet loss of 8%. These factors contribute to the

Test Queries 

Use these test inputs:
Query 1
My mobile internet is very slow in Pune since morning.
Query 2
Customer CUST_1001 says video calls keep dropping in Pune.
Query 3
Customer reports poor browsing speed but device is working fine.

Expected Comparison
Basic Chatbot Output
Likely generic:
•	restart phone
•	check signal
•	move to better coverage area
•	try again later

Telecom Agent Output
Likely grounded:
•	customer is on 5G Unlimited Plus
•	device issue probability is low
•	tower utilization is high
•	latency and packet loss are elevated
•	most likely cause is tower congestion

Lab Explanation
Step 1
“First, we build a basic chatbot. It can understand the complaint, but it cannot verify anything.”
Step 2
“Now we add telecom tools. The same model becomes more powerful because it can fetch real signals.”
Step 3
“This is the shift from chatbot to agent: not just answering, but diagnosing with data.”

LangGraph Version for Conversion
If you want the same lab using LangGraph, the flow can be:
Read Complaint
   ↓
Fetch Customer Profile
   ↓
Fetch Network Metrics
   ↓
Fetch Device Health
   ↓
Diagnose
   ↓
Generate Customer Response
This is better if you want participants to see:
•	structured workflow
•	orchestration
•	state passing
•	enterprise design

Hands-On Exercise for Participants

Modify the agent in one of these ways:

Exercise 1
Add a new tool

In [ ]:
@tool
def get_outage_status(location: str) -> str:
    """Check outage status in a location."""
    data = {
        "location": location,
        "outage_detected": False,
        "planned_maintenance": False
    }
    return json.dumps(data, indent=2)

Exercise 2
Make the agent include:
•	severity level
•	next best action
•	whether to escalate
Exercise 3
Change the scenario from:
•	slow internet
to
•	frequent call drops

•	Basic chatbot = conversational assistant
•	Telecom agent = tool-using diagnostic system
•	In telecom, real value comes when the system can combine:
o	language understanding
o	customer context
o	network signals
o	action-oriented reasoning

What is Chain-of-Thought (CoT)?

•	Instead of jumping to the final answer, the model is instructed to think aloud by breaking the problem into logical reasoning steps.
•	This improves accuracy, interpretability, and often factual correctness, especially for complex or multi-step questions.

In [ ]:
#Create a Tool (Example: ibm_telecom)

from langchain.tools import tool

@tool
def telecom_sales(city: str) -> str:
    """Returns a Telecom report for the given city."""
    return f"IBM Telecom Sales in {city} "

In [21]:
#Initialize GPT Model (via LangChain)

from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1", model_provider="openai")

In [63]:
#Create the ReAct Agent (with Tool Binding)
from langgraph.prebuilt import create_react_agent

tools = [get_weather]
agent_executor = create_react_agent(model=model5, tools=tools)

Chain-of-Thought in Action:

The key here is the phrase:

"Think step-by-step about possible causes."

This triggers CoT reasoning, prompting the model to:
  • Break the problem into smaller steps.
  • Analyze logically.
  • Avoid jumping to conclusions.


In [ ]:
# Run the Agent Without Memory (Single Turn)
input_message = {"role": "user", "content": "Why Temperature has increased by 2 degree in a September Month in the Pune city. Think step-by-step about possible causes."}

response3 = agent_executor.invoke({"messages": [input_message]})
for msg in response3["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Why Temperature has increased by 2 degree September Month in Pune city. Think step-by-step about possible causes.
================================== Ai Message ==================================

A 2°C bump in September in Pune is plausible and usually comes from a mix of monsoon variability and local factors. Here’s a step-by-step way to think about possible causes and how they interact:

1) Define the change
- What exactly rose by 2°C: daytime maximum, nighttime minimum, or daily mean?
- Compared to what baseline: last year, the 1981–2010/1991–2020 climatology, or the month-to-date average?

2) Cloud and rainfall anomaly
- Fewer clouds and a rainfall deficit increase sunshine and surface heating, lifting daytime maxima by 1–4°C.
- Conversely, more clouds and high humidity can keep nights warmer (higher minima) even if days aren’t unusually hot.

3) Monsoon phase and synoptic setup
- Break monsoon or wea

In [29]:
# Run the Agent Without Memory (Single Turn)

input_message = {"role": "user", "content": "Walmart sales increased by 20% this month in chicago city.Think step-by-step about possible causes."}
response = agent_sales_executor.invoke({"messages": [input_message]})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Walmart sales increased by 20% this month in chicago city.Think step-by-step about possible causes.
================================== Ai Message ==================================

To analyze the possible causes of a 20% increase in Walmart sales in Chicago this month, we can consider several factors step-by-step:

### 1. **Seasonal Trends**
   - **Weather Changes**: If the weather has shifted (e.g., colder temperatures), people may be purchasing seasonal items like winter clothing, heaters, or holiday decorations.
   - **Holiday Season**: If this month coincides with a holiday (e.g., Thanksgiving, Christmas, etc.), sales might increase due to holiday shopping.

### 2. **Promotions and Discounts**
   - **Sales Events**: Walmart may have run significant promotions, such as Black Friday sales, clearance events, or special discounts.
   - **Marketing Campaigns**: Aggressive advertising or targeted campaigns

In [17]:
#Use CoT with LangChain or OpenAI
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-4.1", model_provider="openai")

query = {
    "role": "user",
    "content": "Walmart's sales increased by 20% this month. Think step-by-step about possible causes."
}

response = model.invoke([query])
print(response.text())

Certainly! Let’s break down possible causes—step by step—for Walmart’s 20% sales increase this month:

### **1. Economic Factors**
- **Improved Consumer Confidence:** People may be feeling better about their finances, leading to more spending.
- **Inflation:** Price increases could push total sales higher, even if the quantity sold remains flat.

### **2. Company-Specific Actions**
- **Promotions & Discounts:** Walmart might have run special sales, discounts, or “Rollback” promotions.
- **Product Launches:** New product releases, especially electronics, groceries, or exclusive brands, can drive up sales.
- **Expanded E-commerce:** Growth in online ordering, curbside pickup, or expanded delivery services can draw more customers.
- **Store Openings/Remodels:** New or refreshed locations may attract additional foot traffic.

### **3. External Events**
- **Seasonal Effects:** Holidays, events (like back-to-school), or weather (hurricanes, snowstorms) often drive people to stock up.
- **Sti

What is Plan-and-Execute (PnE)?

Plan-and-Execute is a powerful agent pattern that:
•	First plans a sequence of subtasks based on the user request.
•	Then executes each step, often using tools, APIs, or functions.
This improves reliability, transparency, and modularity in AI task solving.

Use Case:
Prompt: “Launch a weekend sale campaign across 3 top-selling categories.”
 
Step 1: Plan
The agent first thinks through all the required steps before doing anything:

PLAN:
To launch a weekend sale campaign across the top 3 categories, the following steps are required:

1. Identify the top 3 selling product categories for the last 30 days.
2. Retrieve current pricing and inventory levels for these categories.
3. Draft a weekend sale campaign with discounts and promo copy.
4. Submit the campaign for manager approval.

Step 2: Execute Each Step
Each planned step is then executed one by one, using appropriate tools or API functions:

✅ Step 1: Identify top categories
🛠️ Tool Call → get_top_categories(last_n_days=30, top_k=3)

Response:
["Groceries", "Electronics", "Personal Care"]

Step 2: Fetch pricing and stock
🛠️ Tool Call → get_pricing_and_inventory(["Groceries", "Electronics", "Personal Care"])
Response:
{
  "Groceries": {"avg_price": "$3.5", "in_stock": "High"},
  "Electronics": {"avg_price": "$120", "in_stock": "Moderate"},
  "Personal Care": {"avg_price": "$8", "in_stock": "High"}
}


✅ Step 3: Create campaign draft
🛠️ Tool Call → create_campaign_draft(...)
Response:
Drafted a weekend promo:  
🛒 Groceries – 15% off  
🔌 Electronics – 10% cashback  
🧼 Personal Care – Buy 1 Get 1 Free  

✅ Step 4: Submit for approval
🛠️ Tool Call → submit_campaign(draft_id="promo_2025_weekend_001")
Response:
✅ Campaign submitted for approval. Awaiting response from Marketing Lead.

✅ Final Response to User:
🎯 Campaign Planning & Execution Complete!

🔹 Categories Selected: Groceries, Electronics, Personal Care  
🔹 Campaign Draft: Discounts + Promo Copy Created  
🔹 Status: Submitted for approval (ID: promo_2025_weekend_001)

In [ ]:
#%%writefile weekend_sale_agent1.py

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain.chat_models import init_chat_model

# ✅ Define your tools with docstrings
@tool
def get_top_categories() -> list:
    """Returns top 3 selling product categories."""
    return ["Groceries", "Electronics", "Personal Care"]

@tool
def get_pricing_and_inventory() -> dict:
    """Returns pricing and inventory levels."""
    return {
        "Groceries": {"avg_price": "$3.5", "in_stock": "High"},
        "Electronics": {"avg_price": "$120", "in_stock": "Moderate"},
        "Personal Care": {"avg_price": "$8", "in_stock": "High"},
    }

@tool
def create_campaign_draft() -> str:
    """Creates a campaign draft with promotions."""
    return """
    Groceries – 15% off
    Electronics – 10% cashback
    Personal Care – Buy 1 Get 1 Free
    """

@tool
def submit_campaign() -> str:
    """Submits the campaign for approval."""
    return " Campaign submitted for approval. ID: promo_2025_weekend_001"

#  Load GPT-4 model (or GPT-3.5 for free tier)
#llm = ChatOpenAI(model="gpt-4", temperature=0.3)

model5 = init_chat_model("gpt-5", model_provider="openai")

# Register tools
tools = [get_top_categories, get_pricing_and_inventory, create_campaign_draft, submit_campaign]

# Create agent using LangGraph
memory = MemorySaver()
agent_executor = create_react_agent(model5, tools, checkpointer=memory)

# Set thread ID
config = {"configurable": {"thread_id": "campaign-001"}}

# Ask agent to plan and execute
input_msg = {"role": "user", "content": "Launch a weekend sale campaign across 3 top-selling categories."}

for step in agent_executor.stream({"messages": [input_msg]}, config, stream_mode="values"):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Launch a weekend sale campaign across 3 top-selling categories.
================================== Ai Message ==================================
Tool Calls:
  get_top_categories (call_0WPNBZhrwXeA9fGq73JSxzRB)
 Call ID: call_0WPNBZhrwXeA9fGq73JSxzRB
  Args:
  get_pricing_and_inventory (call_Ho28tkQuwwQtHVPMWWnOrn11)
 Call ID: call_Ho28tkQuwwQtHVPMWWnOrn11
  Args:
================================= Tool Message =================================
Name: get_pricing_and_inventory

{"Groceries": {"avg_price": "$3.5", "in_stock": "High"}, "Electronics": {"avg_price": "$120", "in_stock": "Moderate"}, "Personal Care": {"avg_price": "$8", "in_stock": "High"}}
================================== Ai Message ==================================
Tool Calls:
  create_campaign_draft (call_V69NhWqDSjJmFAFU8EeMkhtX)
 Call ID: call_V69NhWqDSjJmFAFU8EeMkhtX
  Args:
================================= Tool Message =================

In [66]:
# Ask agent to plan and execute
input_msg = {"role": "user", "content": "Launch a weekend sale campaign across 3 top-selling categories. Create Plan Report for Executives"}

for step in agent_executor.stream({"messages": [input_msg]}, config, stream_mode="values"):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Launch a weekend sale campaign across 3 top-selling categories. Create Plan Report for Executives
================================== Ai Message ==================================
Tool Calls:
  get_top_categories (call_LpqexRYX17QFrwIU7nz1s7Fq)
 Call ID: call_LpqexRYX17QFrwIU7nz1s7Fq
  Args:
  get_pricing_and_inventory (call_qReIfK12t331F19LIlk0iUqv)
 Call ID: call_qReIfK12t331F19LIlk0iUqv
  Args:
================================= Tool Message =================================
Name: get_pricing_and_inventory

{"Groceries": {"avg_price": "$3.5", "in_stock": "High"}, "Electronics": {"avg_price": "$120", "in_stock": "Moderate"}, "Personal Care": {"avg_price": "$8", "in_stock": "High"}}
================================== Ai Message ==================================
Tool Calls:
  create_campaign_draft (call_O5okjHsqyFgPH86f8AUpSOg1)
 Call ID: call_O5okjHsqyFgPH86f8AUpSOg1
  Args:
==============================

================================[1m Human Message [0m=================================

Launch a weekend sale campaign across 3 top-selling categories. Create Plan Report for Executives
==================================[1m Ai Message [0m==================================
Tool Calls:
  get_top_categories (call_LpqexRYX17QFrwIU7nz1s7Fq)
 Call ID: call_LpqexRYX17QFrwIU7nz1s7Fq
  Args:
  get_pricing_and_inventory (call_qReIfK12t331F19LIlk0iUqv)
 Call ID: call_qReIfK12t331F19LIlk0iUqv
  Args:
=================================[1m Tool Message [0m=================================
Name: get_pricing_and_inventory

{"Groceries": {"avg_price": "$3.5", "in_stock": "High"}, "Electronics": {"avg_price": "$120", "in_stock": "Moderate"}, "Personal Care": {"avg_price": "$8", "in_stock": "High"}}
==================================[1m Ai Message [0m==================================
Tool Calls:
  create_campaign_draft (call_O5okjHsqyFgPH86f8AUpSOg1)
 Call ID: call_O5okjHsqyFgPH86f8AUpSOg1
  Args:
=================================[1m Tool Message [0m=================================
Name: create_campaign_draft


    🛒 Groceries – 15% off
    🔌 Electronics – 10% cashback
    🧼 Personal Care – Buy 1 Get 1 Free
    
==================================[1m Ai Message [0m==================================
Tool Calls:
  submit_campaign (call_BnyPm82iZ0OQVFgzOdDZL1Nd)
 Call ID: call_BnyPm82iZ0OQVFgzOdDZL1Nd
  Args:
=================================[1m Tool Message [0m=================================
Name: submit_campaign

✅ Campaign submitted for approval. ID: promo_2025_weekend_001
==================================[1m Ai Message [0m==================================

Executive Plan Report — Weekend Sale Campaign (Top 3 Categories)

Summary
- Campaign ID: promo_2025_weekend_001 (submitted for approval)
- Scope: 3 top-selling categories — Groceries, Electronics, Personal Care
- Proposed window: Fri Sep 12, 12:01 AM to Sun Sep 14, 11:59 PM (local time)
- Promotions (current draft):
  - Groceries: 15% off
  - Electronics: 10% cashback
  - Personal Care: Buy 1 Get 1 Free
- Inventory readiness: Groceries High, Electronics Moderate, Personal Care High
- Average prices: Groceries $3.50, Electronics $120, Personal Care $8.00

Objectives
- Drive weekend traffic, unit velocity, and market share in top categories
- Lift total basket size and cross-category attachment
- Acquire/reactivate customers ahead of Q4

Operational Plan
- Targeting: All shoppers; prioritize loyalty members and lapsed 90-day customers via CRM
- Channels: Homepage hero, category landing pages, app banners, email, push, paid social/search
- Controls:
  - Per-customer limit: 5 redemptions per category
  - Cashback cap: $50 per order for Electronics
  - Exclusions: Doorbusters, preorders, gift cards, Apple/Sony MAP-protected SKUs, tobacco, alcohol where restricted
  - Stacking: No stacking with other sitewide offers; loyalty points accrue as normal
  - Returns: Cashback forfeited on returns; B1G1 returns require paired item or adjusted refund
- Legal/compliance: Promo copy with “while supplies last”, “select SKUs”, geographic restrictions; ADA-compliant creatives

Financial Snapshot (baseline vs projected, weekend only; directional, based on assumptions)
- Baseline weekend revenue (illustrative):
  - Groceries $1.00M, Electronics $0.60M, Personal Care $0.40M (Total $2.00M)
- Inventory note: Electronics stock moderate; ensure replenishment triggers and substitution rules for fast-movers
- Current draft projections (as submitted; assumes vendor funding at 40% for CPG, none for Electronics; halo from non-promoted categories included)
  - Uplift assumptions:
    - Groceries: +20% units, effective discount 15% (7.5% net after 50% vendor funding if secured)
    - Electronics: +12% units, 10% cashback (net 10%)
    - Personal Care: +30% units, effective discount 25% on participating units (net ~15% with 40% vendor funding)
  - Revenue impact on promoted items: approximately +2% to +5% combined
  - Gross profit impact on promoted items: approximately -20% to -40% without halo; net -10% to -15% after halo and vendor funding
  - Halo effects (traffic/basket lift on non-promoted): +4% to +10% revenue on remainder of catalog; GP +$30K to +$75K (assumes $3.0M non-promoted baseline at ~25% margin)
  - Net weekend outcome (range): Revenue +2% to +5%; Gross profit -3% to -12%
- Key sensitivities: Vendor funding secured %, Electronics sell-through at moderate inventory, B1G1 attach rate and mix

Recommendation to improve margin while preserving volume
- Groceries: Move from 15% off to 10% off with $25 basket minimum; secure 40%+ vendor funding
- Electronics: Reduce to 8% cashback, cap $50 per order; feature 50 SKUs with sufficient on-hand; add bundle offers (e.g., laptop + accessory 12% off)
- Personal Care: Change B1G1 to “Second at 50% off” on curated SKUs; require 2+ units
- Expected effect: Similar revenue lift; improves gross profit by ~300–600 bps vs current draft
- Decision: Approve “Optimized Variant” if margin guardrails are required; current draft can proceed if share/traffic is the primary objective

Marketing and Creative
- Messages:
  - Groceries: “Stock up for the weekend — 10–15% off essentials”
  - Electronics: “Limited-time cashback up to $50 on top tech”
  - Personal Care: “Buy more, save more: 50% off your second item”
- Deliverables: 3 hero banners, 12 category tiles, 6 email modules, 8 push variants
- A/B tests: Discount framing (percent vs dollars), urgency badges, order minimums, cashback cap messaging

Measurement and KPIs
- Primary: Incremental revenue, gross profit, contribution margin, campaign ROI
- Secondary: Traffic, conversion rate, AOV, units/order, category attach, new vs returning customer mix, redemption rate, cashback burn, returns
- Diagnostics: Inventory OOS hours, rainchecks/backorders, fraud/abuse rate, coupon leakage
- Attribution: Match-market holdout on 10% of traffic; media geo holdouts

Risks and Mitigations
- Stockouts (Electronics): Prioritize in-stock SKUs; dynamic de-promotion on low OOS thresholds; waitlist with delayed cashback
- Margin compression: Enforce caps/minimum baskets; vendor co-op funding; SKU-level guardrails
- Compliance/MAP: Pre-cleared SKU list; automated MAP monitor
- Fraud/abuse: Per-user caps, device/account velocity checks, cashback issuance after delivery window

Go-Live Checklist
- Confirm promo rules and caps in promo engine
- Verify SKU lists and inventory thresholds; enable back-up SKUs
- QA price tests on top 50 SKUs per category across web/app
- Load creative and schedule email/push; QA links and tracking
- Enable dashboards: real-time sales, GP, OOS, redemption
- Staff CS and social response; publish FAQ
- Final legal review of disclosures

Next Steps (owner • due)
- Finance: Validate funding commitments with top 10 CPG vendors • today
- Merchandising: Lock SKU lists and substitutes • today
- Marketing: Approve creatives; schedule email/push and paid • today
- Engineering: Implement promo rules and QA • today
- Analytics: Stand up dashboards and holdouts • today
- Ops: DC/store readiness, replenishment thresholds • today

Decision asks
- Approve promo structure: Current draft vs Optimized Variant
- Approve weekend window and caps (electronics cashback cap $50; per-customer 5 redemptions)
- Confirm vendor funding targets (≥40% for Groceries/Personal Care)

Status
- Campaign submitted for approval: promo_2025_weekend_001
- Ready to schedule and publish upon executive sign-off

If you’d like, I can:
- Switch to the Optimized Variant and resubmit
- Set the exact start/end times and push live upon approval
- Produce the creative set and the KPI dashboard links for launch day

http://127.0.0.1:8000/mcp?session_id=user001&query=Show%20top%205%20selling%20items%20in%20Pune

Run Server:
uvicorn walmart_mcp_schema:app --reload

In [45]:
import langchain
import langchain_core
print("LangChain version:", langchain.__version__)
print("LangChain Core version:", langchain_core.__version__)

LangChain version: 0.3.26
LangChain Core version: 0.3.66


In [ ]:
from langgraph.graph import StateGraph
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableLambda
from langchain_community.tools.python_repl.tool import PythonREPLTool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import ToolNode, tools_condition

# 1. Define the state
def run_python_code(state):
    tool = PythonREPLTool()
    input_text = state["messages"][-1].content
    result = tool.invoke(input_text)
    return {"messages": [HumanMessage(content=str(result))]}

# 2. Convert tool to OpenAI-compatible function
py_tool = convert_to_openai_tool(run_python_code)

# 3. Create LLM and tool node
llm = ChatOpenAI(model="gpt-4o", temperature=0)
llm_node = ToolNode(llm)

# 4. Create the graph
builder = StateGraph(schema={"messages": list})

builder.add_node("llm", llm_node)
builder.add_node("python", RunnableLambda(run_python_code))

# Conditional routing
builder.set_conditional_entry_point("llm", tools_condition)
builder.add_edge("llm", "python")
builder.add_edge("python", END)

graph = builder.compile()

# 5. Run the agent
result = graph.invoke({
    "messages": [HumanMessage(content="What is (45 * 23) + 129 ?")]
})

# 6. Print final result
final_message = result["messages"][-1].content
print("🤖 Agent Final Response:", final_message)

In [8]:
%%writefile langgraph_python_repl.py

from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda
from langchain_community.tools.python_repl import PythonREPLTool

# Step 1: Define the shared state
class AgentState(dict):
    pass

# Step 2: Define the REPL tool
python_repl = PythonREPLTool()

# Step 3: Define a LangGraph node that uses the REPL
def run_python(state: AgentState) -> AgentState:
    question = state["messages"][-1].content
    print(f"💡 Executing: {question}")
    result = python_repl.invoke(question)
    return {"messages": state["messages"] + [AIMessage(content=result)]}

tool_node = RunnableLambda(run_python)

# Step 4: Build the LangGraph
builder = StateGraph(AgentState)
builder.add_node("python", tool_node)
builder.set_entry_point("python")
builder.set_finish_point("python")
graph = builder.compile()

# Step 5: Run it
result = graph.invoke({"messages": [HumanMessage(content="(45 * 23) + 129")]})
print("🤖 Agent Final Response:", result["messages"][-1].content)

Overwriting langgraph_python_repl.py


In [9]:
%%writefile langgraph_agent2.py

from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langchain_community.tools.python_repl import PythonREPLTool

# Step 1: Define shared state
class AgentState(dict):
    pass

# Step 2: Initialize tools
llm = ChatOpenAI(temperature=0, model="gpt-4o")  # Or use Gemini if preferred
python_tool = PythonREPLTool()

# Step 3: Define logic nodes

# 3a. Reasoning node using LLM
def decide_action(state: AgentState) -> AgentState:
    user_msg = state["messages"][-1].content
    tool_prompt = f"""You are a smart assistant. Decide:

- If the question needs Python calculation, say: "use_tool"
- Else answer directly

Query: {user_msg}
Answer:"""
    decision = llm.invoke([HumanMessage(content=tool_prompt)])
    print(f"🤖 LLM Decision: {decision.content}")
    if "use_tool" in decision.content.lower():
        return {"messages": state["messages"], "use_tool": True}
    return {"messages": state["messages"] + [AIMessage(content=decision.content)], "use_tool": False}

# 3b. Tool execution node
def run_python(state: AgentState) -> AgentState:
    user_input = state["messages"][-1].content
    output = python_tool.invoke(user_input)
    return {"messages": state["messages"] + [AIMessage(content=output)]}

# Step 4: LangGraph setup
builder = StateGraph(AgentState)
builder.add_node("decide", RunnableLambda(decide_action))
builder.add_node("python", RunnableLambda(run_python))

# Conditional routing
def should_use_tool(state: AgentState) -> str:
    return "python" if state.get("use_tool") else END

builder.set_entry_point("decide")
builder.add_conditional_edges("decide", should_use_tool, {"python": "python", END: END})
builder.set_finish_point("python")

graph = builder.compile()

# Step 5: Run agent
result = graph.invoke({"messages": [HumanMessage(content="What is (45 * 23) + 129?")]})
print("🧠 Final Response:", result["messages"][-1].content)

Writing langgraph_agent2.py


In [10]:
import getpass
import os

# Set your OpenAI API key securely if not already set in environment
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("🔐 Enter your OpenAI API key: ")

# Import LangChain's chat model initializer
from langchain.chat_models import init_chat_model

# Initialize GPT-4.1 model from OpenAI
model = init_chat_model(
    model="gpt-4.1",
    model_provider="openai",     # Explicitly tell LangChain to use OpenAI
    temperature=0.0              # Optional: make outputs deterministic
)

# Test the model
response = model.invoke([{"role": "user", "content": "Hi there! Who are you?"}])
print(response.text())

Hello! I’m ChatGPT, an AI language model created by OpenAI. I’m here to help answer your questions, have conversations, and assist with a wide range of topics. How can I help you today?


In [7]:
# Run the Agent Without Memory (Single Turn)

input_message = {"role": "user", "content": "What is the weather in Pune?"}
response = agent_executor.invoke({"messages": [input_message]})

#print(response["messages"][0])

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the weather in Pune?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_t3fgz3OPptrjov5yXnKfq3Nl)
 Call ID: call_t3fgz3OPptrjov5yXnKfq3Nl
  Args:
    city: Pune
================================= Tool Message =================================
Name: get_weather

It's always sunny in Pune with 25°C!
================================== Ai Message ==================================

The weather in Pune is sunny with a temperature of 25°C!


In [17]:
#Add Memory for Multi-Turn Agent

from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

agent_executor = create_react_agent(model=Amodel, tools=tools, checkpointer=memory)

config = {"configurable": {"thread_id": "weather-001"}}

# Turn 1
agent_executor.invoke({"messages": [{"role": "user", "content": "My city is Bangalore"}]}, config)

# Turn 2
response=agent_executor.invoke({"messages": [{"role": "user", "content": "What's the weather where I live?"}]}, config)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My city is Bangalore
================================== Ai Message ==================================

Bangalore, also known as Bengaluru, is the capital city of Karnataka, India. It's famous for its pleasant weather, tech industry, and vibrant culture. Would you like to know the current weather in Bangalore?
================================ Human Message =================================

What's the weather where I live?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_2PkBg10lh8xJvfF4foJ3c1gG)
 Call ID: call_2PkBg10lh8xJvfF4foJ3c1gG
  Args:
    city: Bangalore
================================= Tool Message =================================
Name: get_weather

It's always sunny in Bangalore with 25°C!
================================== Ai Message ==================================

The weather in Bangalore is sunny and pleasant with a tempera

In [3]:
#Create a Tool (Example: walmart_sales)

from langchain.tools import tool

@tool
def walmart_sales(city: str) -> str:
    """Returns a walmart sales report for the given city."""
    return f"Walmart Sales in {city} "

In [27]:
#Create the ReAct Agent (with Tool Binding)
from langgraph.prebuilt import create_react_agent

tools = [walmart_sales]
agent_sales_executor = create_react_agent(model=Amodel, tools=tools)

In [28]:
# Run the Agent Without Memory (Single Turn)

input_message = {"role": "user", "content": "How it is increased by 20% this month in chicago city."}
response = agent_sales_executor.invoke({"messages": [input_message]})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

How it is increased by 20% this month in chicago city.
================================== Ai Message ==================================
Tool Calls:
  walmart_sales (call_98CbfIOSbsMITAIK5j3VCcps)
 Call ID: call_98CbfIOSbsMITAIK5j3VCcps
  Args:
    city: Chicago
================================= Tool Message =================================
Name: walmart_sales

Walmart Sales in Chicago 
================================== Ai Message ==================================

Could you clarify your question? Are you asking about a 20% increase in Walmart sales in Chicago this month, or something else?


In [29]:
#Use CoT with LangChain
query = {
    "role": "user",
    "content": "Walmart's sales increased by 20% this month. Think step-by-step about possible causes."
}
response = Amodel.invoke([query])
print(response.text())

Certainly! Let's think step-by-step about possible causes for Walmart's 20% sales increase this month:

---

### **1. Seasonal Factors**
- **Holidays or Events:** If this month coincided with major holidays (e.g., Thanksgiving, Christmas, or back-to-school season), customers may have increased spending on gifts, decorations, food, and supplies.
- **Weather Changes:** Seasonal shifts, such as colder weather, could drive demand for winter clothing, heating supplies, or holiday-related items.

---

### **2. Promotions and Discounts**
- **Sales Campaigns:** Walmart may have launched aggressive promotions, discounts, or "rollback" pricing on popular items, attracting more customers.
- **Special Events:** Events like "Black Friday," "Cyber Monday," or clearance sales could have boosted foot traffic and online sales.
- **Loyalty Programs:** Enhanced loyalty rewards or exclusive member discounts (e.g., Walmart+ perks) might have incentivized repeat purchases.

---

### **3. Economic Factors**


In [30]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

Use Case:
Prompt: “Launch a weekend sale campaign across 3 top-selling categories.”

Step 1: Plan
The agent first thinks through all the required steps before doing anything:

PLAN:
To launch a weekend sale campaign across the top 3 categories, the following steps are required:

1. Identify the top 3 selling product categories for the last 30 days.
2. Retrieve current pricing and inventory levels for these categories.
3. Draft a weekend sale campaign with discounts and promo copy.
4. Submit the campaign for manager approval.

In [ ]:
%%writefile weekend_sale_agent1.py

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

#  Define your tools with docstrings
@tool
def get_top_categories() -> list:
    """Returns top 3 selling product categories."""
    return ["Groceries", "Electronics", "Personal Care"]

@tool
def get_pricing_and_inventory() -> dict:
    """Returns pricing and inventory levels."""
    return {
        "Groceries": {"avg_price": "$3.5", "in_stock": "High"},
        "Electronics": {"avg_price": "$120", "in_stock": "Moderate"},
        "Personal Care": {"avg_price": "$8", "in_stock": "High"},
    }

@tool
def create_campaign_draft() -> str:
    """Creates a campaign draft with promotions."""
    return """
    🛒 Groceries – 15% off
    🔌 Electronics – 10% cashback
    🧼 Personal Care – Buy 1 Get 1 Free
    """

@tool
def submit_campaign() -> str:
    """Submits the campaign for approval."""
    return "✅ Campaign submitted for approval. ID: promo_2025_weekend_001"

# ✅ Load GPT-4 model (or GPT-3.5 for free tier)
#llm = ChatOpenAI(model="gpt-4", temperature=0.3)

# Register tools
tools = [get_top_categories, get_pricing_and_inventory, create_campaign_draft, submit_campaign]

# Create agent using LangGraph
memory = MemorySaver()
agent_executor = create_react_agent(Amodel, tools, checkpointer=memory)

# Set thread ID
config = {"configurable": {"thread_id": "campaign-001"}}

# Ask agent to plan and execute
input_msg = {"role": "user", "content": "Launch a weekend sale campaign across 3 top-selling categories."}

for step in agent_executor.stream({"messages": [input_msg]}, config, stream_mode="values"):
    step["messages"][-1].pretty_print()

Overwriting weekend_sale_agent1.py


In [5]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection
import numpy as np

In [ ]:
pip install llama-cpp-python

In [9]:
# Use a file-based sqlite database for Milvus Lite
connections.connect(alias="default", uri="my_milvus_data.db")

/opt/anaconda3/lib/python3.11/site-packages/milvus_lite/__init__.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [10]:
# Define fields
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=4)
]

In [11]:
schema = CollectionSchema(fields, description="Simple vector collection")

In [12]:
# Create collection
collection = Collection(name="my_collection", schema=schema)

In [13]:
# Create dummy data
ids = [1, 2, 3, 4, 5]
vectors = np.random.rand(5, 4).tolist()  # 5 vectors of dimension 4

# Insert into collection
collection.insert([ids, vectors])

(insert count: 5, delete count: 0, upsert count: 0, timestamp: 0, success count: 5, err count: 0

In [14]:
collection.create_index(
    field_name="vector",
    index_params={"index_type": "IVF_FLAT", "metric_type": "L2", "params": {"nlist": 10}}
)

Status(code=0, message=: )

In [15]:
collection.load()

In [16]:
query_vectors = np.random.rand(2, 4).tolist()  # 2 query vectors

results = collection.search(
    data=query_vectors,
    anns_field="vector",
    param={"metric_type": "L2", "params": {"nprobe": 5}},
    limit=3,
    output_fields=["id"]
)

# Print results
for i, hits in enumerate(results):
    print(f"Query {i}:")
    for hit in hits:
        print(f" - ID: {hit.entity.get('id')}, Distance: {hit.distance}")

Query 0:
 - ID: 2, Distance: 0.09638828784227371
 - ID: 1, Distance: 0.12075266242027283
 - ID: 3, Distance: 0.6506839990615845
Query 1:
 - ID: 2, Distance: 0.1908564269542694
 - ID: 1, Distance: 0.2642040550708771
 - ID: 3, Distance: 0.5159474611282349


In [22]:
connections.disconnect(alias="default")

In [32]:
#Setup Milvus Lite (Local Vector DB)

from pymilvus import connections

# Connect to Milvus Lite with local storage
connections.connect(alias="ragvector", uri="rag_vectors.db")

connections.disconnect(alias="ragvector")

To build a Milvus Lite direct search with a custom retriever, 
fully compatible with local .db files (no server required).


Milvus Lite + Custom Retriever Example

1️⃣ Install Dependencies

pip install pymilvus sentence-transformers


In [33]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection
from sentence_transformers import SentenceTransformer
import numpy as np

# Use local storage (.db file)
connections.connect(alias="default", uri="my_rag_vectors.db")

In [34]:
# Define fields
fields = [
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=False),
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=384),  # Using MiniLM 384D
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=500)
]

schema = CollectionSchema(fields, description="RAG Collection with Milvus Lite")

# Create collection
collection = Collection(name="rag_collection", schema=schema)

In [35]:
#Generate Embeddings

model = SentenceTransformer("all-MiniLM-L6-v2")  # 384 dimensions

documents = [
    "Generative AI is transforming industries.",
    "Milvus Lite is a local vector database.",
    "LangChain helps build RAG pipelines.",
    "Vector search is important for semantic retrieval."
]

embeddings = model.encode(documents)

In [36]:
#Insert Data into Milvus Lite
ids = list(range(1, len(documents) + 1))
collection.insert([ids, embeddings.tolist(), documents])

(insert count: 4, delete count: 0, upsert count: 0, timestamp: 0, success count: 4, err count: 0

In [37]:
#Create Index & Load Collection

collection.create_index(
    field_name="vector",
    index_params={"index_type": "IVF_FLAT", "metric_type": "L2", "params": {"nlist": 10}}
)

collection.load()

In [38]:
#Define Custom Retriever Function

def retrieve(query, top_k=2):
    query_vec = model.encode([query])

    search_params = {"metric_type": "L2", "params": {"nprobe": 5}}

    results = collection.search(
        data=query_vec,
        anns_field="vector",
        param=search_params,
        limit=top_k,
        output_fields=["id", "text"]
    )

    retrieved = []
    for hits in results:
        for hit in hits:
            retrieved.append({
                "id": hit.entity.get("id"),
                "text": hit.entity.get("text"),
                "distance": hit.distance
            })
    return retrieved

In [39]:
#Run a Query

query = "What is Milvus Lite?"

results = retrieve(query)

for res in results:
    print(f"ID: {res['id']}, Distance: {res['distance']:.4f}, Text: {res['text']}")

ID: 2, Distance: 0.6488, Text: Milvus Lite is a local vector database.
ID: 4, Distance: 1.7359, Text: Vector search is important for semantic retrieval.
